# Lesson 8.3 — Capstone: Analyzer + Tracker Integration (robust velocity layer)
**Module 6 · Unit 8 · Lesson 31**

Wire the analyzer into the tracker: σ_min schedules the damping so joint rates stay bounded through singular regions; the null space carries secondary goals for redundant arms — all while honoring the tool command as closely as geometry allows.

In [1]:
import numpy as np
def dh(th,d,a,al):
    ct,st,ca,sa=np.cos(th),np.sin(th),np.cos(al),np.sin(al)
    return np.array([[ct,-st*ca,st*sa,a*ct],[st,ct*ca,-ct*sa,a*st],[0,sa,ca,d],[0,0,0,1]])
def forward_chain(P,T,q):
    M=np.eye(4); Ms=[M.copy()]
    for i,(th0,d0,a,al) in enumerate(P):
        th,d=(th0+q[i],d0) if T[i]=="R" else (th0,d0+q[i]); M=M@dh(th,d,a,al); Ms.append(M.copy())
    return Ms
def geometric_jacobian(P,T,q):
    Ms=forward_chain(P,T,q); on=Ms[-1][:3,3]; J=np.zeros((6,len(q)))
    for i in range(len(q)):
        z=Ms[i][:3,2]; o=Ms[i][:3,3]
        if T[i]=="R": J[:3,i]=np.cross(z,on-o); J[3:,i]=z
        else: J[:3,i]=z
    return J
def Jv_planar(P,T,q): return geometric_jacobian(P,T,q)[:2,:]
def fk_xy(P,T,q): return forward_chain(P,T,q)[-1][:2,3]
P2=[(0,0,1,0),(0,0,1,0)]; T2=["R","R"]
P3=[(0,0,1,0),(0,0,1,0),(0,0,0.6,0)]; T3=["R","R","R"]   # redundant (2D task, 3 joints)
EPS=0.08   # singularity threshold on sigma_min


In [2]:
def analyze(P,T,q):
    """One SVD -> full capability report (Units 4-6 in one function)."""
    J=Jv_planar(P,T,q); U,S,Vt=np.linalg.svd(J)
    w=float(np.prod(S)); kappa=float(S[0]/max(S[-1],1e-12))
    return {"sigma":S,"w":w,"kappa":kappa,
            "axes":[(U[:,i],float(S[i])) for i in range(len(S))],
            "singular":bool(S[-1]<EPS),"sigma_min":float(S[-1]),"J":J,"U":U,"Vt":Vt}


In [3]:
def dls(J,lam):
    return J.T@np.linalg.inv(J@J.T+lam**2*np.eye(J.shape[0]))
def track_step(P,T,q,xi_d,dt,lam=0.0):
    """Resolve commanded tool twist -> joint rates, integrate open-loop one step."""
    J=Jv_planar(P,T,q)
    qd=(dls(J,lam) if lam>0 else np.linalg.pinv(J))@xi_d
    return q+qd*dt, qd


In [4]:
def schedule_lambda(sigma_min, lam_max=0.1, eps=EPS):
    """lambda^2 = (1-(smin/eps)^2) lam_max^2 inside the band, else 0."""
    if sigma_min>=eps: return 0.0
    return float(np.sqrt((1-(sigma_min/eps)**2))*lam_max)
def velocity_layer(P,T,q,xi_d,z=None):
    rep=analyze(P,T,q); lam=schedule_lambda(rep["sigma_min"])
    J=rep["J"]; Jd=dls(J,lam) if lam>0 else np.linalg.pinv(J)
    qd=Jd@xi_d
    if z is not None:   # null-space secondary objective (redundant arms)
        qd=qd+(np.eye(len(q))-Jd@J)@z
    info={"w":rep["w"],"kappa":rep["kappa"],"sigma_min":rep["sigma_min"],"lambda":lam,"singular":rep["singular"]}
    return qd, info


## Driving toward a singularity: scheduled damping bounds ||qdot||, undamped blows up

In [5]:
checks=[]
def run(scheduled):
    q=np.array([0.4,1.4]); dt=0.01; xi=np.array([0.5,0.0]); peak=0
    for _ in range(300):
        if scheduled: qd,info=velocity_layer(P2,T2,q,xi)
        else: _,qd=track_step(P2,T2,q,xi,dt,lam=0.0)
        peak=max(peak,np.linalg.norm(qd)); q=q+qd*dt
    return peak
pk_d=run(True); pk_u=run(False)
print(f"peak ||qdot||  scheduled-damping={pk_d:.2f}   undamped={pk_u:.2f}")
checks.append(pk_d < pk_u and pk_d < 5.0)

peak ||qdot||  scheduled-damping=4.03   undamped=174.00


## Redundant arm: a null-space secondary velocity leaves the tool twist unchanged

In [6]:
q=np.array([0.3,0.5,-0.4]); xi=np.array([0.2,0.1])
z=np.array([0.0,0.0,1.0])  # secondary joint-velocity preference
qd,info=velocity_layer(P3,T3,q,xi,z=z)
tool_twist=Jv_planar(P3,T3,q)@qd
print("commanded tool twist:",xi,"  achieved:",np.round(tool_twist,4))
checks.append(np.allclose(tool_twist,xi,atol=1e-6))
print("null-space term changed the JOINT motion but not the TOOL twist.")
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

commanded tool twist: [0.2 0.1]   achieved: [0.2 0.1]
null-space term changed the JOINT motion but not the TOOL twist.
All checks passed.
